# Demo 1: HTML Parsing & Section Detection for SEC 10-K Financial Filings

## 1. Theory & Concepts
SEC 10-K filings are annual financial reports required for public companies in the United States. A report is typically hundreds of pages long and contains a substantial amount of noise. For the RAG system to work effectively, we must execute the **Document Ingestion (Preprocessing)** stage, which consists of two sub-steps:
1. **Parsing:** Extracting clean text from complex HTML/XBRL code and removing empty tables, CSS tags, or junk Javascript code.
2. **Section Detection:** Using regular expressions (Regex) as routers to partition the report into core sections (Items) such as:
   * **Item 1:** Business (Business description)
   * **Item 1A:** Risk Factors (Crucial for risk assessment)
   * **Item 7:** MD&A (Management's Discussion and Analysis of Financial Condition and Results of Operations - contains detailed data explanations)
   * **Item 8:** Financial Statements and Supplementary Data (Contains balance sheets, income statements, cash flows, and notes)

### Data Flow:
* **Input:** Raw HTML files (.html) downloaded directly from the SEC EDGAR database.
* **Output:** Clean text sections clearly separated by Item with their corresponding metadata.
* **Purpose of the Output:** These clean text sections will be transferred directly to the next stage: **Chunking**. They are not fed directly into the LLM because the size of a single Item is still too large for the model's context window (Context Window) and would dilute information.

## 2. Initialize Mock HTML Source Code
We will build a miniature HTML code snippet that simulates the actual structure of a 10-K report file.

In [ ]:
html_content = """
<html>
<head><style>.header { color: blue; }</style></head>
<body>
    <div class="header">REGISTRATION STATEMENT UNDER THE SECURITIES ACT OF 1933</div>
    
    <!-- SIMULATE ITEM 1A -->
    <h2 style="text-align:center;">Item 1A. Risk Factors</h2>
    <p>Our business is subject to significant risks. First, global supply chain constraints and component shortages, especially in semiconductors, could materially affect our shipments.</p>
    <p>Second, intense competition in the smartphone and cloud service markets could reduce our gross margin.</p>
    
    <!-- SIMULATE ITEM 7 -->
    <h2 style="text-align:center;">Item 7. Management's Discussion and Analysis of Financial Condition</h2>
    <p>During fiscal year 2023, our net sales increased by 8% to $383,285 million compared to fiscal year 2022.</p>
    <p>Operating income was $114,301 million, representing a strong performance driven by services growth.</p>
    
    <!-- SIMULATE ITEM 8 -->
    <h2>Item 8. Financial Statements and Supplementary Data</h2>
    <table border="1">
        <tr><th>Year Ended</th><th>Net Income</th></tr>
        <tr><td>September 30, 2023</td><td>$96,995 million</td></tr>
        <tr><td>September 24, 2022</td><td>$99,803 million</td></tr>
    </table>
</body>
</html>
"""
print("=== INPUT (Raw HTML) ===")
print(html_content[:300] + "\n...[truncated]...")

## 3. Parsing: Extracting Text with BeautifulSoup4
Remove all HTML and CSS tags, keeping the raw text structure.

In [ ]:
from bs4 import BeautifulSoup
import re

print("[Log] Running BeautifulSoup4 to parse HTML...")
soup = BeautifulSoup(html_content, "html.parser")

# Remove tags that do not contain readable content
for tag in soup(["style", "script", "head", "title", "meta"]):
    tag.decompose()

clean_text = soup.get_text(separator="\n")
print("[SUCCESS] Converted to clean text successfully!")
print("\n=== OUTPUT (Raw text after parsing) ===")
print(clean_text)

## 4. Section Detection: Segmenting Text using Regex
We design regular expressions (Regex) to detect the boundaries of each Item and separate them into independent text blocks.

In [ ]:
# Define Regex boundaries for target Items
patterns = {
    "Item 1A": r"item\s*1a[.\s]*risk\s*factors",
    "Item 7": r"item\s*7[.\s]*management's\s*discussion",
    "Item 8": r"item\s*8[.\s]*financial\s*statements"
}

print("[Log] Starting scan for item boundaries...")
positions = []
for item_name, pat in patterns.items():
    match = re.search(pat, clean_text, re.IGNORECASE)
    if match:
        start_pos = match.start()
        positions.append((item_name, start_pos))
        print(f"  - Detected '{item_name}' at character: {start_pos}")

# Sort detected positions by appearance order
positions.sort(key=lambda x: x[1])

# Segment text strings based on identified positions
sections = {}
for i in range(len(positions)):
    item_name, start = positions[i]
    # End point is start of next Item, or end of file
    end = positions[i+1][1] if i + 1 < len(positions) else len(clean_text)
    
    # Slice string and normalize whitespace
    section_text = clean_text[start:end].strip()
    sections[item_name] = section_text
    print(f"[Log] Successfully extracted '{item_name}' (Length: {len(section_text)} characters)")

print("\n" + "="*40)
print("OUTPUT SEGMENTATION RESULTS")
print("="*40)
for item_name, text in sections.items():
    print(f"\n>>> {item_name} <<<")
    print(text[:300] + "...")

## 5. Conclusion and Data Handover
The output of this stage consists of clean text strings segmented by target Item. 
These clean text strings, along with metadata labels such as `ticker = 'AAPL'`, `year = 2023`, `section = 'Item 7'`, will be handed over directly as input for **Step 2: Chunking** to split them into chunks of uniform size with overlapping context.